# Lesson 11: Subgraph Architecture — Reusable Arithmetic Engine

## Objective
Compile a reusable **arithmetic engine** as a subgraph and embed it inside a parent graph as a single node — demonstrating LangGraph modularity.

## Limitation of Lesson 10
- ❌ All logic was in one flat graph — validation, routing, compute all mixed together
- ❌ No way to reuse the arithmetic logic in another graph
- ❌ No separation of concerns

## What is NEW in Lesson 11?
- ✅ **Subgraph**: a compiled graph embedded as a node inside another graph
- ✅ **State compatibility**: inner and outer graphs have compatible state fields
- ✅ **Modular architecture**: the arithmetic engine is a black box to the outer graph
- ✅ **Composability**: subgraphs can themselves contain subgraphs

## Theory: Subgraphs

In LangGraph, a compiled graph (`CompiledGraph`) can be **added as a node** inside another graph:

```python
subgraph = inner_builder.compile()
outer_builder.add_node("engine", subgraph)  # Subgraph as node!
```

When the outer graph reaches the `"engine"` node, it hands off execution to the subgraph, which runs to completion, then control returns to the outer graph.

### State Handoff Rules
- The subgraph receives the **outer state** when invoked as a node
- The subgraph's state fields must be a **subset** of (or compatible with) the outer state
- The subgraph returns a partial update that the outer graph merges normally

### Why subgraphs exist architecturally
- **Reuse**: the arithmetic engine can be used by many parent graphs
- **Encapsulation**: inner logic is hidden from the outer graph
- **Testing**: test the subgraph independently
- **Composition**: build complex agents from simple modules


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import operator
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Literal, Annotated
from IPython.display import Image, display

print("✓ Imports successful")
print("  No new imports needed — subgraph is a compile() pattern")


In [ ]:
# ── Cell 2: Shared State Schema ──────────────────────────────────────────────
# IMPORTANT: Both the subgraph and outer graph must use compatible state fields
# We use the SAME State class for both here (simplest case)

class State(TypedDict):
    a: int
    b: int
    operation: str
    result: Optional[float]
    history: Annotated[list, operator.add]
    validated: bool    # Set by outer graph's validation step

print("✓ Shared state schema defined")


In [ ]:
# ── Cell 3: Build the Inner Arithmetic Engine (Subgraph) ─────────────────────
# This is a self-contained arithmetic engine
# It only knows about: a, b, operation, result, history

def inner_add(state: State) -> dict:
    res = state["a"] + state["b"]
    entry = f"[engine] {state['a']} + {state['b']} = {res}"
    print(f"  {entry}")
    return {"result": float(res), "history": [entry]}

def inner_multiply(state: State) -> dict:
    res = state["a"] * state["b"]
    entry = f"[engine] {state['a']} × {state['b']} = {res}"
    print(f"  {entry}")
    return {"result": float(res), "history": [entry]}

def inner_divide(state: State) -> dict:
    res = state["a"] / state["b"]
    entry = f"[engine] {state['a']} ÷ {state['b']} = {res:.4f}"
    print(f"  {entry}")
    return {"result": res, "history": [entry]}

def engine_router(state: State) -> Literal["inner_add", "inner_multiply", "inner_divide"]:
    return {"add": "inner_add", "multiply": "inner_multiply", "divide": "inner_divide"}[state["operation"]]

# ── Build the subgraph ───────────────────────────────────────────────────────
engine_builder = StateGraph(State)
engine_builder.add_node("inner_add",      inner_add)
engine_builder.add_node("inner_multiply", inner_multiply)
engine_builder.add_node("inner_divide",   inner_divide)

engine_builder.add_conditional_edges(START, engine_router, {
    "inner_add":      "inner_add",
    "inner_multiply": "inner_multiply",
    "inner_divide":   "inner_divide",
})
engine_builder.add_edge("inner_add",      END)
engine_builder.add_edge("inner_multiply", END)
engine_builder.add_edge("inner_divide",   END)

# Compile the subgraph — it's now a CompiledGraph object
arithmetic_engine = engine_builder.compile()

print("✓ Arithmetic engine subgraph compiled")
print(f"  Type: {type(arithmetic_engine)}")


In [ ]:
from IPython.display import Image, displaydisplay(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# ── Cell 5: Build the Outer Graph ────────────────────────────────────────────
# The outer graph handles:
# 1. Input validation (before calling the engine)
# 2. Logging (after the engine completes)
# The engine itself is a BLACK BOX to the outer graph

def outer_validate(state: State) -> dict:
    print(f"  [outer:validate] Checking: a={state['a']}, b={state['b']}, op='{state['operation']}'")
    if state["operation"] not in ("add", "multiply", "divide"):
        raise ValueError(f"Unknown operation: {state['operation']}")
    if state["operation"] == "divide" and state["b"] == 0:
        raise ValueError("Division by zero not allowed")
    print(f"  [outer:validate] ✓ Input valid")
    return {"validated": True}

def outer_logger(state: State) -> dict:
    print(f"  [outer:logger]  Result = {state['result']}")
    entry = f"[logger] Completed: result={state['result']}"
    return {"history": [entry]}

outer_builder = StateGraph(State)
outer_builder.add_node("validate",          outer_validate)
outer_builder.add_node("arithmetic_engine", arithmetic_engine)  # ← SUBGRAPH AS NODE
outer_builder.add_node("logger",            outer_logger)

outer_builder.add_edge(START,               "validate")
outer_builder.add_edge("validate",          "arithmetic_engine")  # Hand off to engine
outer_builder.add_edge("arithmetic_engine", "logger")
outer_builder.add_edge("logger",            END)

parent_graph = outer_builder.compile()
print("✓ Parent graph with embedded subgraph compiled")
print("  arithmetic_engine is a black-box node from parent's perspective")


In [ ]:
# ── Cell 8: Run Parent Graph ─────────────────────────────────────────────────
print("="*50)
print("Test: 20 ÷ 4")
print("="*50)
state = {"a": 20, "b": 4, "operation": "divide", "result": None, "history": [], "validated": False}
out = parent_graph.invoke(state)
print(f"\n  Result: {out['result']}")
print(f"  History: {out['history']}")


In [ ]:
# ── Cell 9: More Tests ───────────────────────────────────────────────────────
for a, b, op in [(7, 8, "multiply"), (15, 9, "add"), (100, 25, "divide")]:
    state = {"a":a,"b":b,"operation":op,"result":None,"history":[],"validated":False}
    out = parent_graph.invoke(state)
    print(f"  {a} {op:>8} {b} = {out['result']}")


In [ ]:
# ── Cell 10: Use Subgraph Independently ──────────────────────────────────────
# The subgraph can ALSO be used standalone — full modularity
print("Using arithmetic_engine subgraph directly (no outer graph):")
state = {"a": 9, "b": 3, "operation": "multiply", "result": None, "history": [], "validated": False}
out = arithmetic_engine.invoke(state)
print(f"  Direct invocation: {out['result']}")


## Subgraph Architecture Diagram

```
┌──────────────────────────────────────────────────────┐
│  PARENT GRAPH                                        │
│                                                      │
│  START → validate → ┌─────────────────────────────┐ │
│                     │  ARITHMETIC ENGINE SUBGRAPH  │ │
│                     │  START → router → add/mul/   │ │
│                     │          divide → END        │ │
│                     └─────────────────────────────┘ │
│                     → logger → END                   │
└──────────────────────────────────────────────────────┘
```

## Key Concepts

### Subgraph as Node
```python
compiled_subgraph = inner_builder.compile()
outer_builder.add_node("engine", compiled_subgraph)  # ← This is all you need
```

### State Compatibility
- Subgraph state fields must be a **subset** of the outer state
- OR: use input/output schema mapping for non-overlapping state schemas

### `xray=True` visualization
```python
graph.get_graph(xray=True).draw_mermaid_png()
# Shows internal structure of embedded subgraphs
```

## Summary

| | Flat Graph | Subgraph Architecture |
|--|-----------|----------------------|
| Reusability | No | Yes |
| Testability | Together | Independently |
| Complexity | Grows unbounded | Encapsulated |

## Limitations → What Lesson 12 Solves
- ❌ Still one graph doing all the work
- ❌ For complex systems, you need multiple **agents** each with their own LLM and tools
- ❌ **Lesson 12**: Multi-agent pattern — Supervisor routes to Worker agents
